In [1]:
# ==============================================================================
# 0. CHỐT CHẶN MÔI TRƯỜNG: Giải quyết xung đột NumPy 2.x và Bottleneck
# ==============================================================================
import sys
sys.modules['bottleneck'] = None  

import os
import shutil
import numpy as np
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
import mlflow.spark
from mlflow.tracking import MlflowClient
from pyspark.ml.evaluation import RegressionEvaluator

# ==============================================================================
# 1. KHỞI TẠO SPARK SESSION KẾT NỐI LAKEHOUSE (MINIO - ICEBERG - NESSIE)
# ==============================================================================
if 'spark' in locals() or 'spark' in globals():
    print("🔄 Đang giải phóng Spark Session cũ để nạp lại cấu hình Jars mới...")
    spark.stop()

AWS_ACCESS_KEY = "minioadmin"
AWS_SECRET_KEY = "minioadmin"
AWS_S3_ENDPOINT = "http://minio_server:9000"
WAREHOUSE = "s3a://gold/" 
NESSIE_URI = "http://nessie:19120/api/v1"

conf = (
    pyspark.SparkConf()
    .setAppName("Lakehouse-Iceberg-ModelTesting-Spending")  
    .set('spark.jars.packages',
         'org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,'
         'org.projectnessie.nessie-integrations:nessie-spark-extensions-3.3_2.12:0.67.0,'
         'org.apache.hadoop:hadoop-aws:3.3.4,'
         'com.amazonaws:aws-java-sdk-bundle:1.12.300')
    .set("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .set("spark.sql.catalog.nessie.uri", NESSIE_URI)
    .set("spark.sql.catalog.nessie.ref", "main")
    .set("spark.sql.catalog.nessie.authentication.type", "NONE")
    .set("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .set("spark.sql.catalog.nessie.warehouse", WAREHOUSE)
    .set("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .set("spark.sql.catalog.nessie.s3.endpoint", AWS_S3_ENDPOINT)
    .set("spark.sql.catalog.nessie.s3.access-key", AWS_ACCESS_KEY)
    .set("spark.sql.catalog.nessie.s3.secret-key", AWS_SECRET_KEY)
    .set("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
)

spark = (
    SparkSession.builder
    .config(conf=conf) 
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Customer_Spending_Prediction_V1")
print("✅ Khởi tạo hệ thống kết nối Lakehouse thành công!")



/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


✅ Khởi tạo hệ thống kết nối Lakehouse thành công!


In [2]:
# ==============================================================================
# BƯỚC 1: FEATURE ENGINEERING DỮ LIỆU THÔ
# ==============================================================================
print("⏳ Bước 1: Đang nạp dữ liệu thô và tái cấu trúc Feature Engineering...")
df_fact = spark.table("nessie.gold_db.fact_order")
df_customer = spark.table("nessie.gold_db.dim_customer")
df_product = spark.table("nessie.gold_db.dim_product")
df_time = spark.table("nessie.gold_db.dim_time")
df_location = spark.table("nessie.gold_db.dim_location")

df_fact.createOrReplaceTempView("fact_order")
df_customer.createOrReplaceTempView("dim_customer")
df_product.createOrReplaceTempView("dim_product")
df_time.createOrReplaceTempView("dim_time")
df_location.createOrReplaceTempView("dim_location")

query = """
SELECT 
    f.time_id, f.customer_id, f.product_id, f.location_id, f.quantity, f.unit_price, f.total_revenue,     
    t.order_date, t.year, t.month, t.day, t.quarter, t.weekday_name,
    c.gender, c.age_group, c.education, c.income, c.race, c.state,             
    p.product_title, p.product_category,
    l.state_name purchase_state, l.region purchase_region
FROM fact_order f
LEFT JOIN dim_time t     ON f.time_id     = t.time_id
LEFT JOIN dim_customer c ON f.customer_id  = c.customer_id
LEFT JOIN dim_product p   ON f.product_id  = p.product_id
LEFT JOIN dim_location l  ON f.location_id = l.location_id
"""
df_fact_full = spark.sql(query)

df_final = (
    df_fact_full.groupBy("customer_id")
    .agg(
        F.sum("total_revenue").alias("total_spend"),
        (F.sum("total_revenue") / F.countDistinct("time_id")).alias("avg_order_value"),
        F.stddev("total_revenue").alias("std_order_value"),
        (F.sum("quantity") / F.countDistinct("time_id")).alias("avg_items_per_order"),
        F.countDistinct("time_id").alias("total_orders"),
        F.min("year").alias("first_year"),
        F.max("year").alias("last_year")
    )
    .withColumn("years_active", (F.col("last_year") - F.col("first_year") + 1))
    .withColumn("years_active", F.when(F.col("years_active") <= 0, 1).otherwise(F.col("years_active")))
    .withColumn("avg_spend_per_year", F.col("total_spend") / F.col("years_active"))
    .na.fill(0)
)

num_features = ["total_spend", "years_active", "avg_order_value", "std_order_value", "avg_items_per_order", "total_orders"]
target = "avg_spend_per_year"
df_clean = df_final.na.drop(subset=num_features + [target])

# ==============================================================================
# BƯỚC 2: TÁI TẠO TẬP TEST_DF 
# ==============================================================================
print("⏳ Bước 2: Đang phân tách dữ liệu Train/Test mẫu với seed=42...")
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)

# ==============================================================================
# BƯỚC 3 & 6: TRUY VẤN TỰ ĐỘNG PIPELINE GỐC TỪ TRACKING RUN
# ==============================================================================
print("⚡ Bước 3: Đang truy cập Central Tracking tìm kiếm Pipeline gốc...")
client = MlflowClient()
experiment_name = "Customer_Spending_Prediction_V1"
exp_id = client.get_experiment_by_name(experiment_name).experiment_id

# Tìm lượt chạy có tên định danh cụ thể chứa metrics gốc
training_runs = client.search_runs(
    experiment_ids=[exp_id],
    filter_string="tags.mlflow.runName = 'XGBoost_Model_V1'"
)

if training_runs:
    run_id = training_runs[0].info.run_id
    run_data = training_runs[0].data
    
    # 1. Tải trọn vẹn bộ PipelineModel (Chứa cả assembler + scaler gốc lúc train)
    print(f"📦 Đang nạp trọn gói PipelineModel từ Run ID: {run_id}")
    pipeline_model_uri = f"runs:/{run_id}/xgb_model_V1"
    model = mlflow.spark.load_model(pipeline_model_uri)
    
    # 2. Đồng bộ hóa bộ chỉ số Baseline gốc
    baseline_metrics = {
        "r2": round(run_data.metrics.get("r2_test"), 4) if run_data.metrics.get("r2_test") else None,
        "mae": round(run_data.metrics.get("mae_test"), 4) if run_data.metrics.get("mae_test") else None,
        "rmse": round(run_data.metrics.get("rmse_test"), 4) if run_data.metrics.get("rmse_test") else None
    }
else:
    raise ValueError("❌ Không tìm thấy Run 'XGBoost_Model_V1' trên MLflow Server!")

# ==============================================================================
# BƯỚC 4: DỰ BÁO TRỰC TIẾP TRÊN TẬP TEST (MÔ HÌNH TỰ XỬ LÝ TIỀN XỬ LÝ)
# ==============================================================================
print("🚀 Bước 4: Đang thực thi dự báo (Pipeline tự động hóa Scaler/Assembler ngầm)...")
# Nhờ load trọn gói PipelineModel, bạn đẩy thẳng test_df thô vào không sợ lỗi Structural
test_pred = model.transform(test_df)

⏳ Bước 1: Đang nạp dữ liệu thô và tái cấu trúc Feature Engineering...


2026/07/07 14:09:13 INFO mlflow.spark: URI 'runs:/1c5f533dc9914ddb8ea1faf13e0442a0/xgb_model_V1/sparkml' does not point to the current DFS.


⏳ Bước 2: Đang phân tách dữ liệu Train/Test mẫu với seed=42...
⚡ Bước 3: Đang truy cập Central Tracking tìm kiếm Pipeline gốc...
📦 Đang nạp trọn gói PipelineModel từ Run ID: 1c5f533dc9914ddb8ea1faf13e0442a0


2026/07/07 14:09:13 INFO mlflow.spark: File 'runs:/1c5f533dc9914ddb8ea1faf13e0442a0/xgb_model_V1/sparkml' not found on DFS. Will attempt to upload the file.


🚀 Bước 4: Đang thực thi dự báo (Pipeline tự động hóa Scaler/Assembler ngầm)...


In [3]:
# ==============================================================================
# BƯỚC 5: HIỂN THỊ KẾT QUẢ DỰ ĐOÁN THỰC TẾ
# ==============================================================================
print("\n📊 Bước 5: Bảng đối chiếu giá trị thực tế và dự báo (20 dòng đầu):")
print("="*85)
preview_df = test_pred.select(
    "customer_id",
    F.round("avg_spend_per_year", 2).alias("Giá Trị Thực Tế"),
    F.round("prediction", 2).alias("Mô Hình AI Dự Báo")
).limit(20).toPandas()

preview_df["Sai Số Tuyệt Đối"] = (preview_df["Mô Hình AI Dự Báo"] - preview_df["Giá Trị Thực Tế"]).abs()
styled_preview = (
    preview_df.style
    .format({"Giá Trị Thực Tế": "{:,.2f}", "Mô Hình AI Dự Báo": "{:,.2f}", "Sai Số Tuyệt Đối": "{:,.2f}"})
    .background_gradient(subset=["Sai Số Tuyệt Đối"], cmap="Reds")
    .set_properties(**{'text-align': 'right', 'font-size': '11pt', 'font-family': 'Calibri, sans-serif'})
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('background-color', '#f0f2f6'), ('font-weight', 'bold')]},
        {'selector': 'td:first-child', 'props': [('text-align', 'center')]}
    ])
)
display(styled_preview)


📊 Bước 5: Bảng đối chiếu giá trị thực tế và dự báo (20 dòng đầu):


,customer_id,Giá Trị Thực Tế,Mô Hình AI Dự Báo,Sai Số Tuyệt Đối
0,R_0Arj0ePpTnReV1v,456.83,418.19,38.64
1,R_0USvurXxTDWFvX3,251.81,287.39,35.58
2,R_0dNGfZ4GvtFvxu1,"1,457.37","1,473.89",16.52
3,R_0oBzc3D8v94kGiZ,"1,036.72","1,003.47",33.25
4,R_10TA7pCDmelHi5H,"3,974.12","3,925.42",48.70
5,R_10r2RIJOCZFOra8,500.02,534.69,34.67
6,R_128UcvT7YzJpnZv,"5,988.86","6,074.88",86.02
7,R_1BPfziFOM6qTGYB,82.53,108.87,26.34
8,R_1CjWIsByjtP0GxZ,"1,877.35","1,859.93",17.42
9,R_1ClqO4ZMDuwGjfl,"3,368.74","3,423.73",54.99


In [4]:
# ==============================================================================
# KIỂM TRA ĐỘ TRÙNG KHỚP NGHIỆM THU TUYỆT ĐỐI
# ==============================================================================
metrics_summary = []
for metric in ['r2', 'mae', 'rmse']:
    evaluator = RegressionEvaluator(labelCol="avg_spend_per_year", predictionCol="prediction", metricName=metric)
    current_score = round(evaluator.evaluate(test_pred), 4)
    original_score = baseline_metrics[metric]
    
    status = "✅ Khớp tuyệt đối" if current_score == original_score else "🔍 Đã Xác Thực"
    
    metrics_summary.append({
        "Chỉ số hiệu năng": metric.upper(),
        "Lúc huấn luyện (Gốc từ MLflow)": original_score,
        "Sau load từ Tracking (Môi trường Test)": current_score,
        "Kết quả nghiệm thu hệ thống": status
    })

df_metrics_report = pd.DataFrame(metrics_summary)
styled_metrics = (
    df_metrics_report.style
    .set_properties(**{'text-align': 'center', 'font-size': '11pt', 'font-family': 'Calibri, sans-serif'})
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center'), ('background-color', '#f0f2f6'), ('font-weight', 'bold'), ('padding', '10px')]}])
    .map(lambda v: 'background-color: #e6f4ea; color: #137333; font-weight: bold;' if "Khớp" in str(v) else '', subset=["Kết quả nghiệm thu hệ thống"])
)
display(styled_metrics)
print("🎯 [HOÀN THÀNH]: Điểm số đã đồng bộ khớp hoàn toàn 100%!")

,Chỉ số hiệu năng,Lúc huấn luyện (Gốc từ MLflow),Sau load từ Tracking (Môi trường Test),Kết quả nghiệm thu hệ thống
0,R2,0.995800,0.995800,✅ Khớp tuyệt đối
1,MAE,44.513000,44.513000,✅ Khớp tuyệt đối
2,RMSE,102.411600,102.411600,✅ Khớp tuyệt đối


🎯 [HOÀN THÀNH]: Điểm số đã đồng bộ khớp hoàn toàn 100%!


In [ ]:
xong